#  Autonomous SecOps AI Agent - Phase 1: Data Ingestion

> **Dataset:** RT-IoT2022 (UCI Machine Learning Repository ID: 942)  
> **Objective:** Load IoT network telemetry flow records, verify schema and dimensions, inspect missing values, and validate target class counts.

## 1. Environment & Library Imports

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")

Pandas Version: 3.1.1
NumPy Version:  2.5.5


## 2. Locate and Load Dataset
We check the local `data/` directory first for instant loading. If missing, we automatically fetch it via `ucimlrepo`.

In [2]:
def get_dataset_path():
    candidate_paths = [
        "data/RT_IOT2022.csv",
        "../data/RT_IOT2022.csv",
        "../../data/RT_IOT2022.csv",
        os.path.join(os.getcwd(), "data", "RT_IOT2022.csv"),
        os.path.abspath(os.path.join(os.getcwd(), "..", "data", "RT_IOT2022.csv")),
        "/Users/akshith/Desktop/SecOps_AI_Agent/data/RT_IOT2022.csv",
    ]
    # Search up directory hierarchy in case kernel cwd is in a subfolder or root
    curr = os.path.abspath(os.getcwd())
    for _ in range(5):
        candidate_paths.append(os.path.join(curr, "data", "RT_IOT2022.csv"))
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent

    for path in candidate_paths:
        if os.path.exists(path):
            return os.path.abspath(path)

    # Fallback to ucimlrepo fetch only if local file is truly not found
    print(
        f"Local file not found across candidates (cwd: {os.getcwd()}), fetching via ucimlrepo (ID: 942)..."
    )
    from ucimlrepo import fetch_ucirepo

    repo = fetch_ucirepo(id=942)
    df_remote = pd.concat([repo.data.features, repo.data.targets], axis=1)
    save_dir = "data" if os.path.basename(os.getcwd()) != "notebooks" else "../data"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "RT_IOT2022.csv")
    df_remote.to_csv(save_path, index=False)
    return os.path.abspath(save_path)


data_path = get_dataset_path()
print(f"Loading dataset from: {data_path} ...")
df = pd.read_csv(data_path)

# Drop redundant raw index column if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print(
    f"[OK] Dataset successfully loaded! Shape: {df.shape[0]:,} rows × {df.shape[1]} columns"
)

Loading dataset from: /Users/akshith/Personal_Projects/SecOps_AI_Agent/data/RT_IOT2022.csv ...
[OK] Dataset successfully loaded! Shape: 123,117 rows × 84 columns


## 3. Dataset Dimensions & Preview
Inspect the first few network flow telemetry records.

In [3]:
print(f"Total Network Flow Records: {df.shape[0]:,}")
print(f"Total Features (Excluding Target): {df.shape[1] - 1}")
df.head()

Total Network Flow Records: 123,117
Total Features (Excluding Target): 83


,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,bwd_pkts_per_sec,flow_pkts_per_sec,down_up_ratio,fwd_header_size_tot,fwd_header_size_min,fwd_header_size_max,bwd_header_size_tot,bwd_header_size_min,bwd_header_size_max,flow_FIN_flag_count,flow_SYN_flag_count,flow_RST_flag_count,fwd_PSH_flag_count,bwd_PSH_flag_count,flow_ACK_flag_count,fwd_URG_flag_count,bwd_URG_flag_count,flow_CWR_flag_count,flow_ECE_flag_count,fwd_pkts_payload.min,fwd_pkts_payload.max,fwd_pkts_payload.tot,fwd_pkts_payload.avg,fwd_pkts_payload.std,bwd_pkts_payload.min,bwd_pkts_payload.max,bwd_pkts_payload.tot,bwd_pkts_payload.avg,bwd_pkts_payload.std,flow_pkts_payload.min,flow_pkts_payload.max,flow_pkts_payload.tot,flow_pkts_payload.avg,flow_pkts_payload.std,fwd_iat.min,fwd_iat.max,fwd_iat.tot,fwd_iat.avg,fwd_iat.std,bwd_iat.min,bwd_iat.max,bwd_iat.tot,bwd_iat.avg,bwd_iat.std,flow_iat.min,flow_iat.max,flow_iat.tot,flow_iat.avg,flow_iat.std,payload_bytes_per_second,fwd_subflow_pkts,bwd_subflow_pkts,fwd_subflow_bytes,bwd_subflow_bytes,fwd_bulk_bytes,bwd_bulk_bytes,fwd_bulk_packets,bwd_bulk_packets,fwd_bulk_rate,bwd_bulk_rate,active.min,active.max,active.tot,active.avg,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,0.281148,0.156193,0.437341,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0.0,33.0,76.0,8.444444,13.115936,0.0,23.0,32.0,6.4,9.555103,0.0,33.0,108.0,7.714286,11.618477,761.985779,2.972918e+07,3.201160e+07,4.001450e+06,1.040307e+07,4438.877106,1.511694e+06,2.026391e+06,506597.757339,680406.147126,761.985779,2.972918e+07,3.201160e+07,2.462431e+06,8.199747e+06,3.373777,3.0,1.666667,25.333333,10.666667,0.0,0.0,0.0,0.0,0.0,0.0,2.282415e+06,2.282415e+06,2.282415e+06,2.282415e+06,0.0,2.972918e+07,2.972918e+07,2.972918e+07,2.972918e+07,0.0,64240,26847,502,MQTT_Publish
1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,0.282277,0.156821,0.439097,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0.0,33.0,76.0,8.444444,13.115936,0.0,23.0,32.0,6.4,9.555103,0.0,33.0,108.0,7.714286,11.618477,247.001648,2.985528e+07,3.188358e+07,3.985448e+06,1.046346e+07,4214.048386,1.576436e+06,1.876261e+06,469065.248966,741351.686212,247.001648,2.985528e+07,3.188358e+07,2.452583e+06,8.242459e+06,3.387323,3.0,1.666667,25.333333,10.666667,0.0,0.0,0.0,0.0,0.0,0.0,2.028307e+06,2.028307e+06,2.028307e+06,2.028307e+06,0.0,2.985528e+07,2.985528e+07,2.985528e+07,2.985528e+07,0.0,64240,26847,502,MQTT_Publish
2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,0.280164,0.155647,0.435811,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0.0,33.0,74.0,8.222222,12.852799,0.0,21.0,30.0,6.0,8.689074,0.0,33.0,104.0,7.428571,11.229866,283.956528,2.984215e+07,3.212405e+07,4.015507e+06,1.044238e+07,2456.903458,1.476049e+06,2.013770e+06,503442.466259,660344.360027,283.956528,2.984215e+07,3.212405e+07,2.471081e+06,8.230593e+06,3.237450,3.0,1.666667,24.666667,10.000000,0.0,0.0,0.0,0.0,0.0,0.0,2.281904e+06,2.281904e+06,2.281904e+06,2.281904e+06,0.0,2.984215e+07,2.984215e+07,2.984215e+07,2.984215e+07,0.0,64240,26847,502,MQTT_Publish
3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,0.281593,0.156440,0.438033,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0.0,33.0,74.0,8.222222,12.852799,0.0,21.0,30.0,6.0,8.689074,0.0,33.0,104.0,7.428571,11.229866,288.963318,2.991377e+07,3.196106e+07,3.995133e+06,1.048253e+07,3933.906555,1.551892e+06,1.883784e+06,470946.013927,724569.317911,288.963318,2.991377e+07,3.196106e+07,2.458543e+06,8.257786e+06,3.253959,3.0,1.666667,24.666667,10.000000,0.0,0.0,0.0,0.0,0.0,0.0,2.047288e+06,2.047288e+06,2.047288e+06,2.047288e+06,0.0,2.991377e+07,2.991377e+07,2.991377e+07,2.991377e+07,0.0,64240,26847,502,MQTT_Publish
4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,0.282111,0.156728,0.438839,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0.0,33.0,76.0,8.444444,13.115936,0.0,23.0,32.0,6.4,9.555103,0.0,33.0,108.0,7.714286

## 4. Data Quality & Missing Value Inspection

In [4]:
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()
print(f"Total Missing / NaN Values in Entire Dataset: {total_nulls}")
if total_nulls > 0:
    print("Columns with missing values:")
    print(null_counts[null_counts > 0])
else:
    print("[OK] The dataset has zero missing values. Highly complete!")
# Data types summary
print("\nFeature Data Types:")
print(df.dtypes.value_counts())

Total Missing / NaN Values in Entire Dataset: 0
[OK] The dataset has zero missing values. Highly complete!

Feature Data Types:
float64    56
int64      25
str         3
Name: count, dtype: int64


## 5. Target Class Distribution Analysis
The dataset contains 12 verified classes covering DoS/DDoS attacks, reconnaissance/port scans, MITM/brute force, and benign smart-home IoT traffic.

In [5]:
# Compute counts and percentages
counts = df["Attack_type"].value_counts()
percentages = (df["Attack_type"].value_counts(normalize=True) * 100).round(3)


# Categorize broad threat types
def categorize_threat(name):
    if name in ["DOS_SYN_Hping", "DDOS_Slowloris"]:
        return "DoS / DDoS Attack"
    elif name.startswith("NMAP_"):
        return "Reconnaissance / Scan"
    elif name in ["ARP_poisioning", "Metasploit_Brute_Force_SSH"]:
        return "MITM / Brute Force"
    elif name in ["Thing_Speak", "MQTT_Publish", "Wipro_bulb"]:
        return "Benign IoT Traffic"
    return "Other"


summary_df = pd.DataFrame(
    {
        "Count": counts,
        "Percentage (%)": percentages,
        "Threat Category": [categorize_threat(idx) for idx in counts.index],
    }
)

print(summary_df)

                            Count  Percentage (%)        Threat Category
Attack_type                                                             
DOS_SYN_Hping               94659          76.885      DoS / DDoS Attack
Thing_Speak                  8108           6.586     Benign IoT Traffic
ARP_poisioning               7750           6.295     MITM / Brute Force
MQTT_Publish                 4146           3.368     Benign IoT Traffic
NMAP_UDP_SCAN                2590           2.104  Reconnaissance / Scan
NMAP_XMAS_TREE_SCAN          2010           1.633  Reconnaissance / Scan
NMAP_OS_DETECTION            2000           1.624  Reconnaissance / Scan
NMAP_TCP_scan                1002           0.814  Reconnaissance / Scan
DDOS_Slowloris                534           0.434      DoS / DDoS Attack
Wipro_bulb                    253           0.205     Benign IoT Traffic
Metasploit_Brute_Force_SSH     37           0.030     MITM / Brute Force
NMAP_FIN_SCAN                  28           0.023  

## 6. Categorical vs. Numerical Telemetry Breakdown

In [6]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
for col in [c for c in cat_cols if c != "Attack_type"]:
    print(f"Unique values in {col}: {df[col].unique().tolist()}")

print(f"\nNumerical Telemetry Features: {len(num_cols)} features")
print("Sample Numerical Features:", num_cols[:10])

Categorical Columns (3): ['proto', 'service', 'Attack_type']
Unique values in proto: ['tcp', 'udp', 'icmp']
Unique values in service: ['mqtt', '-', 'http', 'dns', 'ntp', 'ssl', 'dhcp', 'irc', 'ssh', 'radius']

Numerical Telemetry Features: 81 features
Sample Numerical Features: ['id.orig_p', 'id.resp_p', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec']


## 7. Key Findings & Next Steps
1. **Volume & Integrity:** 123,117 network flow records with 0 missing values.
2. **Class Imbalance:** `DOS_SYN_Hping` represents ~76.9% of traffic, while minority attacks like `Metasploit_Brute_Force_SSH` (37 records) and `NMAP_FIN_SCAN` (28 records) are rare.
3. **Next Step:** Feature encoding, scaling, train/test split, and multi-class intrusion classification model training.